# N4 IR-Only YOLO11l Specialist - Kaggle T4x2

Runs the N4 YOLO11l segmentation specialist using only IR training images and labels. This large IR-only specialist is intended both as a stronger target-domain reference and as the future IR member of an ensemble with the N3 EO+IR YOLO11l model. The notebook generates the N4 training/eval folders inside `/kaggle/working`, runs smoke first, then full, and evaluates `best.pt` on IR-only and EO+IR.


In [ ]:
from pathlib import Path

RUN_STAGE = "smoke"  # smoke first, then full

GITHUB_REPO = "https://github.com/AyushPanchal/domain-adaptation-segmentation.git"
WORK_DIR = Path("/kaggle/working")
REPO_DIR = WORK_DIR / "domain-adaptation-segmentation"
DATASET_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/ayushbpanchal/indraeye-seg"),
    Path("/kaggle/input/indraeye-seg"),
]
DATASET_ROOT = next((path for path in DATASET_ROOT_CANDIDATES if path.exists()), DATASET_ROOT_CANDIDATES[0])

EXPERIMENT_ID = "N4"
EXPERIMENT_NAME = "N4_ir_only_yolo11l"
METHOD_NAME = "ir_only_yolo11l"
GENERATED_ROOT = WORK_DIR / "generated" / "n4_ir_only_yolo11l"
DATASET_YAML = "data/manifests/dataset_yamls/kaggle_n4_ir_only_yolo11l.yaml"
EVAL_IR_YAML = "data/manifests/dataset_yamls/kaggle_n4_eval_ir.yaml"
EVAL_EO_IR_YAML = "data/manifests/dataset_yamls/kaggle_n4_eval_eo_ir.yaml"
EXPERIMENT_CONFIG = "configs/experiments/n4_kaggle_ir_only_yolo11l.yaml"

YOLO_DEVICE = "0,1"
YOLO_EVAL_DEVICE = "0"
YOLO_BATCH = "8"  # safer for YOLO11l on Kaggle T4x2
YOLO_WORKERS = "2"
YOLO_PATIENCE = "25"
YOLO_RESUME = "auto"
YOLO_EPOCHS = "1" if RUN_STAGE == "smoke" else "100"

OUTPUT_ROOT = WORK_DIR / "runs" / f"kaggle_n4_ir_only_yolo11l_{RUN_STAGE}"
REPORT_DIR = Path(f"reports/tables/kaggle_n4_ir_only_yolo11l_{RUN_STAGE}")

assert RUN_STAGE in {"smoke", "full"}, RUN_STAGE
print("RUN_STAGE:", RUN_STAGE)
print("DATASET_ROOT:", DATASET_ROOT)
print("GENERATED_ROOT:", GENERATED_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("YOLO_DEVICE/BATCH/EPOCHS:", YOLO_DEVICE, YOLO_BATCH, YOLO_EPOCHS)


## Helpers

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
from datetime import datetime
from PIL import Image

def stage(title):
    print("\n" + "=" * 92)
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {title}")
    print("=" * 92)

def run_live(command, cwd=None, env=None):
    stage("RUN: " + " ".join(map(str, command)))
    process = subprocess.Popen(
        list(map(str, command)), cwd=str(cwd) if cwd else None, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
    return_code = process.wait()
    print(f"\n[return_code] {return_code}")
    if return_code != 0:
        raise RuntimeError(f"Command failed with return code {return_code}: {command}")

def count_files(path, suffixes):
    return sum(1 for item in Path(path).rglob("*") if item.suffix.lower() in suffixes)

def image_files(path):
    suffixes = {".jpg", ".jpeg", ".png"}
    return sorted(item for item in Path(path).iterdir() if item.suffix.lower() in suffixes)

def show_tail(path, lines=60):
    path = Path(path)
    stage(f"TAIL: {path}")
    if not path.exists():
        print("missing")
        return
    text = path.read_text(encoding="utf-8", errors="replace").splitlines()
    print("\n".join(text[-lines:]))


## Stage 1 - Clone Code Repo

In [ ]:
stage("Stage 1 - Clone or update code repo")
if REPO_DIR.exists() and (REPO_DIR / ".git").exists():
    run_live(["git", "pull"], cwd=REPO_DIR)
elif REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
    run_live(["git", "clone", GITHUB_REPO, str(REPO_DIR)])
else:
    run_live(["git", "clone", GITHUB_REPO, str(REPO_DIR)])
run_live(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR)


## Stage 2 - Verify Dataset

In [ ]:
stage("Stage 2 - Verify source dataset")
required_dirs = [
    DATASET_ROOT / "ir/images/train",
    DATASET_ROOT / "ir/labels/train",
    DATASET_ROOT / "ir/images/val",
    DATASET_ROOT / "ir/labels/val",
    DATASET_ROOT / "eo/images/val",
    DATASET_ROOT / "eo/labels/val",
]
for path in required_dirs:
    print(path, "OK" if path.exists() else "MISSING")
    if not path.exists():
        raise FileNotFoundError(path)
print("IR train images:", count_files(DATASET_ROOT / "ir/images/train", {".jpg", ".jpeg", ".png"}))
print("IR train labels:", count_files(DATASET_ROOT / "ir/labels/train", {".txt"}))
print("IR val images:", count_files(DATASET_ROOT / "ir/images/val", {".jpg", ".jpeg", ".png"}))
print("EO val images:", count_files(DATASET_ROOT / "eo/images/val", {".jpg", ".jpeg", ".png"}))


## Stage 3 - Generate IR-Only Dataset

In [ ]:
stage("Stage 3 - Generate N4 IR-only YOLO11l train/eval folders")

def reset_dir(path):
    path = Path(path)
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

def copy_split_pairs(name, sources):
    out_images = GENERATED_ROOT / "images" / name
    out_labels = GENERATED_ROOT / "labels" / name
    reset_dir(out_images)
    reset_dir(out_labels)
    total = 0
    for prefix, image_dir, label_dir in sources:
        for img in image_files(image_dir):
            label = label_dir / f"{img.stem}.txt"
            if not label.exists():
                continue
            dst_stem = f"{prefix}_{img.stem}"
            shutil.copy2(img, out_images / f"{dst_stem}{img.suffix.lower()}")
            shutil.copy2(label, out_labels / f"{dst_stem}.txt")
            total += 1
    print(f"{name}: {total} image/label pairs")

copy_split_pairs("train", [
    ("ir", DATASET_ROOT / "ir/images/train", DATASET_ROOT / "ir/labels/train"),
])
copy_split_pairs("eval_ir", [
    ("ir", DATASET_ROOT / "ir/images/val", DATASET_ROOT / "ir/labels/val"),
])
copy_split_pairs("eval_eo_ir", [
    ("eo", DATASET_ROOT / "eo/images/val", DATASET_ROOT / "eo/labels/val"),
    ("ir", DATASET_ROOT / "ir/images/val", DATASET_ROOT / "ir/labels/val"),
])

print("Generated root:", GENERATED_ROOT)
print("Train IR images:", count_files(GENERATED_ROOT / "images/train", {".jpg", ".jpeg", ".png"}))
print("Eval IR images:", count_files(GENERATED_ROOT / "images/eval_ir", {".jpg", ".jpeg", ".png"}))
print("Eval EO+IR images:", count_files(GENERATED_ROOT / "images/eval_eo_ir", {".jpg", ".jpeg", ".png"}))


## Stage 4 - Write YAMLs

In [ ]:
stage("Stage 4 - Write training/eval YAMLs")

def class_block():
    return """nc: 12
names:
  0: Bicycle
  1: Bus
  2: Car
  3: Cargo trike
  4: Ignore
  5: Motorcycle
  6: Person
  7: Rickshaw
  8: Small truck
  9: Tractor
  10: Truck
  11: Van
"""

yaml_paths = [REPO_DIR / DATASET_YAML, REPO_DIR / EVAL_IR_YAML, REPO_DIR / EVAL_EO_IR_YAML]
for yaml_path in yaml_paths:
    yaml_path.parent.mkdir(parents=True, exist_ok=True)

(REPO_DIR / DATASET_YAML).write_text(f"""path: {GENERATED_ROOT.as_posix()}
train: images/train
val: images/eval_ir

{class_block()}""", encoding="utf-8")
(REPO_DIR / EVAL_IR_YAML).write_text(f"""path: {GENERATED_ROOT.as_posix()}
train: images/train
val: images/eval_ir

{class_block()}""", encoding="utf-8")
(REPO_DIR / EVAL_EO_IR_YAML).write_text(f"""path: {GENERATED_ROOT.as_posix()}
train: images/train
val: images/eval_eo_ir

{class_block()}""", encoding="utf-8")

experiment_config_path = REPO_DIR / EXPERIMENT_CONFIG
experiment_config_path.parent.mkdir(parents=True, exist_ok=True)
experiment_config_path.write_text(f"""id: N4
name: ir_only_yolo11l
method: ir_only_yolo11l
model: yolo11l-seg.pt
dataset: {DATASET_YAML}
train_modality: IR
test_modality: IR
augmentation: none
epochs: 100
imgsz: 640
batch: auto
seed: 42
""", encoding="utf-8")

for path in [REPO_DIR / DATASET_YAML, REPO_DIR / EVAL_IR_YAML, REPO_DIR / EVAL_EO_IR_YAML, experiment_config_path]:
    print(f"\n--- {path} ---")
    print(path.read_text())


## Stage 5 - Install Dependencies And Check T4x2

In [ ]:
stage("Stage 5 - Install dependencies")
run_live([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], cwd=REPO_DIR)

stage("Stage 5 - GPU check")
import torch
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
print("device_count:", torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    print(index, torch.cuda.get_device_name(index))
if torch.cuda.device_count() < 2:
    raise RuntimeError("Expected Kaggle T4x2. Enable GPU T4x2 before training.")


## Stage 6 - Train And Evaluate

In [ ]:
stage("Stage 6 - Training")
env = os.environ.copy()
env["PYTHONPATH"] = str(REPO_DIR / "src")

resume_args = []
if YOLO_RESUME == "auto":
    resume_args = ["--resume-if-available"]
elif YOLO_RESUME.lower() in {"1", "true", "yes"}:
    resume_args = ["--resume"]

train_command = [
    sys.executable, "-m", "domain_adaptation_segmentation.training.run_experiment",
    "--config", EXPERIMENT_CONFIG,
    "--output-root", str(OUTPUT_ROOT),
    "--device", YOLO_DEVICE,
    "--epochs", YOLO_EPOCHS,
    "--batch", YOLO_BATCH,
    "--workers", YOLO_WORKERS,
    "--patience", YOLO_PATIENCE,
    *resume_args,
]
run_live(train_command, cwd=REPO_DIR, env=env)

collect_command = [
    sys.executable, "-m", "domain_adaptation_segmentation.training.collect_results",
    "--runs-root", str(OUTPUT_ROOT),
    "--output-dir", str(REPORT_DIR),
]
run_live(collect_command, cwd=REPO_DIR, env=env)

best_model = OUTPUT_ROOT / "experiments" / EXPERIMENT_NAME / "ultralytics" / "train" / "weights" / "best.pt"
if not best_model.exists():
    raise FileNotFoundError(best_model)

eval_root = OUTPUT_ROOT / "evaluations"
for eval_name, eval_yaml in [("eval_ir", EVAL_IR_YAML), ("eval_eo_ir", EVAL_EO_IR_YAML)]:
    eval_command = [
        sys.executable, "-m", "domain_adaptation_segmentation.training.evaluate_model",
        "--model", str(best_model),
        "--data", eval_yaml,
        "--output-root", str(eval_root),
        "--name", eval_name,
        "--device", YOLO_EVAL_DEVICE,
        "--imgsz", "640",
        "--batch", YOLO_BATCH,
        "--workers", YOLO_WORKERS,
    ]
    run_live(eval_command, cwd=REPO_DIR, env=env)

run_dir = OUTPUT_ROOT / "experiments" / EXPERIMENT_NAME
show_tail(run_dir / "stdout.log", lines=80)


## Stage 7 - Print Results

In [ ]:
stage("Stage 7 - Results")
import pandas as pd

run_dir = OUTPUT_ROOT / "experiments" / EXPERIMENT_NAME
status_path = run_dir / "status.json"
results_path = run_dir / "results.csv"
summary_path = REPO_DIR / REPORT_DIR / "summary_results.csv"

if status_path.exists():
    status = json.loads(status_path.read_text(encoding="utf-8"))
    print(json.dumps(status, indent=2)[:4000])

if results_path.exists():
    df = pd.read_csv(results_path)
    df.columns = [column.strip() for column in df.columns]
    display(df.tail())

if summary_path.exists():
    print("Summary table:")
    display(pd.read_csv(summary_path))

eval_rows = []
for eval_name in ["eval_ir", "eval_eo_ir"]:
    metrics_path = OUTPUT_ROOT / "evaluations" / eval_name / "metrics.json"
    if not metrics_path.exists():
        print("missing", metrics_path)
        continue
    payload = json.loads(metrics_path.read_text(encoding="utf-8"))
    metrics = payload.get("metrics", {})
    row = {"eval": eval_name}
    for key in [
        "metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)", "metrics/mAP50-95(B)",
        "metrics/precision(M)", "metrics/recall(M)", "metrics/mAP50(M)", "metrics/mAP50-95(M)",
    ]:
        row[key] = metrics.get(key)
    eval_rows.append(row)

if eval_rows:
    print("Post-training best.pt evaluations:")
    display(pd.DataFrame(eval_rows))


## Stage 8 - Package Results

In [ ]:
stage("Stage 8 - Package outputs")
from IPython.display import FileLink, display

bundle_dir = WORK_DIR / f"{RUN_STAGE}_n4_artifacts"
if bundle_dir.exists():
    shutil.rmtree(bundle_dir)
bundle_dir.mkdir(parents=True, exist_ok=True)

if OUTPUT_ROOT.exists():
    shutil.copytree(OUTPUT_ROOT, bundle_dir / OUTPUT_ROOT.name)
if (REPO_DIR / REPORT_DIR).exists():
    shutil.copytree(REPO_DIR / REPORT_DIR, bundle_dir / "tables")

archive_base = WORK_DIR / f"{RUN_STAGE}_n4_results"
archive_path = shutil.make_archive(str(archive_base), "zip", bundle_dir)
print("Result zip:", archive_path)
print("Download through this link or the Kaggle Output/Files panel:")
display(FileLink(archive_path))
